In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from ripser import ripser
from persim import plot_diagrams

SEED = 42
np.random.seed(SEED)

In [ ]:
# ── LOAD & PREPARE ────────────────────────────────────────────────────────────
df = pd.read_csv('../data/MGKDB_unified_parameters_20260426_154819.csv')
param_cols = list(df.select_dtypes(include='number').columns)
df_params = df[param_cols].dropna()

scaler = MinMaxScaler()
X = scaler.fit_transform(df_params).astype(np.float32)
print(f"Shape: {X.shape}  ({X.shape[0]} samples × {X.shape[1]} dims)")

# Subsample for TDA — Vietoris-Rips is O(n^2) in memory
N_TDA = 2000
idx = np.random.choice(len(X), N_TDA, replace=False)
X_tda = X[idx]
print(f"Using {N_TDA} random points for persistent homology")

Shape: (24563, 13)  (24563 samples × 13 dims)
Using 2000 random points for persistent homology


: 

## 1. Persistent homology (Vietoris-Rips) — full 13D

In [ ]:
# H0 = connected components, H1 = loops/cycles, H2 = voids
result_13d = ripser(X_tda, maxdim=2)
dgms_13d   = result_13d['dgms']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

plot_diagrams(dgms_13d, ax=axes[0], show=False)
axes[0].set_title('Persistence diagram — 13D (H0, H1, H2)')

# Barcode for H1 (most informative for loop/cluster structure)
h1 = dgms_13d[1]
h1_finite = h1[h1[:, 1] < np.inf]
lifetimes  = h1_finite[:, 1] - h1_finite[:, 0]
order      = np.argsort(lifetimes)[::-1]
h1_sorted  = h1_finite[order]

for i, (b, d) in enumerate(h1_sorted[:30]):
    axes[1].plot([b, d], [i, i], color='steelblue', linewidth=2, alpha=0.7)
axes[1].set_xlabel('Filtration radius'); axes[1].set_ylabel('H1 feature (sorted by lifetime)')
axes[1].set_title(f'H1 barcode — top 30 of {len(h1_finite)} loops (13D)')

plt.tight_layout(); plt.show()
print(f"H0 features: {len(dgms_13d[0])}, H1: {len(h1_finite)}, H2: {len(dgms_13d[2][dgms_13d[2][:,1]<np.inf])}")

## 2. Persistence entropy & Betti numbers vs filtration radius

In [ ]:
def betti_curve(dgm, radii):
    """Count features alive at each radius."""
    finite = dgm[dgm[:, 1] < np.inf]
    return np.array([(( finite[:, 0] <= r) & (finite[:, 1] > r)).sum() for r in radii])

def persistence_entropy(dgm):
    finite = dgm[dgm[:, 1] < np.inf]
    L = finite[:, 1] - finite[:, 0]
    L = L[L > 0]
    p = L / L.sum()
    return -np.sum(p * np.log(p + 1e-12))

radii = np.linspace(0, 1.5, 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for dim, color, label in [(0, 'steelblue', 'H0 (components)'),
                           (1, 'darkorange', 'H1 (loops)'),
                           (2, 'green',      'H2 (voids)')]:
    axes[0].plot(radii, betti_curve(dgms_13d[dim], radii), color=color, label=label)
axes[0].set_xlabel('Filtration radius'); axes[0].set_ylabel('Betti number')
axes[0].set_title('Betti curves vs filtration radius (13D)')
axes[0].legend()

for dim, color in [(0, 'steelblue'), (1, 'darkorange'), (2, 'green')]:
    dgm = dgms_13d[dim]
    finite = dgm[dgm[:, 1] < np.inf]
    L = finite[:, 1] - finite[:, 0]
    axes[1].hist(L, bins=40, alpha=0.6, color=color, label=f'H{dim}', edgecolor='none')
axes[1].set_xlabel('Lifetime (death − birth)'); axes[1].set_ylabel('Count')
axes[1].set_title('Lifetime distributions')
axes[1].legend()

plt.tight_layout(); plt.show()

for dim in range(3):
    pe = persistence_entropy(dgms_13d[dim])
    print(f"H{dim} persistence entropy: {pe:.3f}")

## 3. How many PCA dims preserve topology? (PH on PCA projections)

In [ ]:
from persim import bottleneck

pca_full = PCA(random_state=SEED).fit(X)
dims_to_test = [2, 3, 4, 5, 6, 8, 10, 13]

bn_h0, bn_h1 = [], []
for d in dims_to_test:
    Z = pca_full.transform(X_tda)[:, :d]
    dgms_d = ripser(Z, maxdim=1)['dgms']
    bn_h0.append(bottleneck(dgms_13d[0], dgms_d[0]))
    bn_h1.append(bottleneck(dgms_13d[1], dgms_d[1]))
    print(f"d={d:2d}  BN_H0={bn_h0[-1]:.4f}  BN_H1={bn_h1[-1]:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dims_to_test, bn_h0, marker='o', label='H0 bottleneck distance', color='steelblue')
ax.plot(dims_to_test, bn_h1, marker='s', label='H1 bottleneck distance', color='darkorange')
ax.set_xlabel('PCA dims kept'); ax.set_ylabel('Bottleneck distance to 13D')
ax.set_title('Topological distortion vs PCA dimensionality\n(lower = topology better preserved)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Mapper graph — topological skeleton of the data

In [ ]:
# Minimal Mapper: filter = PC1, cover = overlapping intervals, cluster = single-linkage
from sklearn.cluster import AgglomerativeClustering
from collections import defaultdict

def mapper(X, filter_vals, n_intervals=10, overlap=0.4):
    """Returns nodes (list of point-index sets) and edges."""
    fmin, fmax = filter_vals.min(), filter_vals.max()
    step = (fmax - fmin) / n_intervals
    width = step * (1 + overlap)
    nodes, node_members = [], []
    for i in range(n_intervals):
        lo = fmin + i * step - (width - step) / 2
        hi = lo + width
        mask = (filter_vals >= lo) & (filter_vals <= hi)
        pts_idx = np.where(mask)[0]
        if len(pts_idx) < 2:
            continue
        n_clusters = max(1, min(5, len(pts_idx) // 10))
        labels = AgglomerativeClustering(n_clusters=n_clusters).fit_predict(X[pts_idx])
        for c in np.unique(labels):
            nodes.append(pts_idx[labels == c])
    edges = []
    for i in range(len(nodes)):
        for j in range(i+1, len(nodes)):
            if len(np.intersect1d(nodes[i], nodes[j])) > 0:
                edges.append((i, j))
    return nodes, edges

Z_pc = pca_full.transform(X_tda)
filter_vals = Z_pc[:, 0]  # PC1 as filter function

nodes, edges = mapper(X_tda, filter_vals, n_intervals=12, overlap=0.4)
print(f"Mapper graph: {len(nodes)} nodes, {len(edges)} edges")

# Layout: x = mean filter value of node, y = mean PC2
node_x = np.array([filter_vals[n].mean() for n in nodes])
node_y = np.array([Z_pc[n, 1].mean() for n in nodes])
node_size = np.array([len(n) for n in nodes])

fig, ax = plt.subplots(figsize=(10, 6))
for i, j in edges:
    ax.plot([node_x[i], node_x[j]], [node_y[i], node_y[j]], 'grey', lw=0.8, alpha=0.5, zorder=1)
sc = ax.scatter(node_x, node_y, s=node_size*2, c=node_x,
                cmap='viridis', zorder=2, edgecolors='k', linewidths=0.3)
plt.colorbar(sc, ax=ax, label='Mean PC1 (filter value)')
ax.set_xlabel('Mean PC1'); ax.set_ylabel('Mean PC2')
ax.set_title('Mapper graph (filter=PC1, cover=12 intervals, 40% overlap)\nNode size ∝ cluster size')
plt.tight_layout(); plt.show()

## 5. Persistent homology on parameter subsets — which dims drive topology?

In [ ]:
# Drop one parameter at a time; measure H1 bottleneck distance to full 13D
# Large distance = that parameter was topologically important
bn_drop = {}
for i, col in enumerate(param_cols):
    keep = [j for j in range(X_tda.shape[1]) if j != i]
    dgms_drop = ripser(X_tda[:, keep], maxdim=1)['dgms']
    bn_drop[col] = bottleneck(dgms_13d[1], dgms_drop[1])
    print(f"Drop {col:20s}  BN_H1={bn_drop[col]:.4f}")

cols_sorted = sorted(bn_drop, key=bn_drop.get, reverse=True)
vals_sorted = [bn_drop[c] for c in cols_sorted]

fig, ax = plt.subplots(figsize=(len(param_cols)*0.7+2, 4))
ax.bar(cols_sorted, vals_sorted, color='steelblue', alpha=0.8, edgecolor='grey')
ax.set_ylabel('H1 bottleneck distance to full 13D')
ax.set_title('Topological importance of each parameter\n(drop-one analysis — higher = more topologically critical)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()